### Load Libraries

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 500)

from scipy.interpolate import interp1d
import allensdk.brain_observatory.behavior.behavior_project_cache as bpc

from importlib.metadata import version
print('allensdk version 2.10.2 or higher is required, you have {} installed'.format(version("allensdk")))

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="Ignoring the following cached namespace")

In [ ]:
output_dir = '/groups/zhang/home/zhangl5/Allen_Brain'

### Data Table

In [ ]:
bc = bpc.VisualBehaviorOphysProjectCache.from_s3_cache(cache_dir=output_dir)

behavior_session_table = bc.get_behavior_session_table()
ophys_session_table = bc.get_ophys_session_table()
experiment_table = bc.get_ophys_experiment_table()

#print number of items in each table for all imaging and behavioral sessions
print('Number of behavior sessions = {}'.format(len(behavior_session_table)))
print('Number of ophys sessions = {}'.format(len(ophys_session_table)))
print('Number of ophys experiments = {}'.format(len(experiment_table)))

#print number of items in each table with Mesoscope imaging
print('Number of behavior sessions with Mesoscope = {}'.format(len(behavior_session_table[behavior_session_table.project_code.isin(['VisualBehaviorMultiscope'])])))
print('Number of ophys sessions with Mesoscope = {}'.format(len(ophys_session_table[ophys_session_table.project_code.isin(['VisualBehaviorMultiscope'])])))
print('Number of ophys experiments with Mesoscope = {}'.format(len(experiment_table[experiment_table.project_code.isin(['VisualBehaviorMultiscope'])])))


### Select One Session

In [ ]:
# get all Sst experiments for ophys session 4
selected_experiment_table = experiment_table[(experiment_table.cre_line=='Sst-IRES-Cre')&
                        (experiment_table.session_number==4) &
                        (experiment_table.prior_exposures_to_image_set==0)]
print('Number of experiments: {}'.format(len(selected_experiment_table)))

# select first experiment from the table to look at in more detail.
# Note that python enumeration starts at 0.
ophys_experiment_id = selected_experiment_table.index[0]
dataset = bc.get_behavior_ophys_experiment(ophys_experiment_id)

In [ ]:
cell_specimen_ids = dataset.cell_specimen_table.index.values # a list of all cell ids
cell_specimen_id = cell_specimen_ids[5] # let's pick 6th cell
print('Cell specimen id = {}'.format(cell_specimen_id)) # print id

In [ ]:
# plot dff and events traces overlaid from the cell selected above
fig, ax = plt.subplots(1,1, figsize = (20,10))
ax.plot(dataset.ophys_timestamps, dataset.dff_traces.loc[cell_specimen_id, 'dff'])
ax.plot(dataset.ophys_timestamps, dataset.events.loc[cell_specimen_id, 'events'])
ax.set_xlabel('time (seconds)')
ax.set_ylabel('trace magnitude')
ax.set_title('Cell specimen id = {}'.format(cell_specimen_id), fontsize = 20)
ax.legend(['dff', 'events'], fontsize = 20)


In [ ]:
# pull the image stimuli from the stimulus table
stimulus_presentations = dataset.stimulus_presentations[
    dataset.stimulus_presentations.stimulus_block_name.str.contains('change_detection')]

# create a list of all unique stimuli presented in this experiment
unique_stimuli = [stimulus for stimulus in stimulus_presentations['image_name'].unique()]

# create a colormap with each unique image having its own color
colormap = {image_name: sns.color_palette()[image_number] for image_number, image_name in enumerate(np.sort(unique_stimuli))}
colormap['omitted'] = (1,1,1) # set omitted stimulus to white color

# add the colors for each image to the stimulus presentations table in the dataset
stimulus_presentations['color'] = stimulus_presentations['image_name'].map(lambda image_name: colormap[image_name])

In [ ]:
# function to plot dff traces
def plot_dff_trace(ax, cell_specimen_id, initial_time, final_time):
    '''
        ax: axis on which to plot
        cell_specimen_id: id of the cell to plot
        intial_time: initial time to plot from
        final_time: final time to plot to
    '''
    #create a dataframe using dff trace from one seleted cell
    data = {'dff': dataset.dff_traces.loc[cell_specimen_id].dff,
        'timestamps': dataset.ophys_timestamps}
    df = pd.DataFrame(data)
    dff_trace_sample = df.query('timestamps >= @initial_time and timestamps <= @final_time')
    ax.plot(
        dff_trace_sample['timestamps'],
        dff_trace_sample['dff']/dff_trace_sample['dff'].max()
    )

# function to plot events traces
def plot_events_trace(ax, cell_specimen_id, initial_time, final_time):
    # create a dataframe using events trace from one seleted cell
    data = {'events': dataset.events.loc[cell_specimen_id].events,
        'timestamps': dataset.ophys_timestamps}
    df = pd.DataFrame(data)
    events_trace_sample = df.query('timestamps >= @initial_time and timestamps <= @final_time')
    ax.plot(
        events_trace_sample['timestamps'],
        events_trace_sample['events']/events_trace_sample['events'].max()
    )
# function to plot running speed
def plot_running(ax, initial_time, final_time):
    running_sample = dataset.running_speed.query('timestamps >= @initial_time and timestamps <= @final_time')
    ax.plot(
        running_sample['timestamps'],
        running_sample['speed']/running_sample['speed'].max(),
        '--',
        color = 'gray',
        linewidth = 1
    )
# function to plot pupil diameter
def plot_pupil(ax, initial_time, final_time):
    pupil_sample = dataset.eye_tracking.query('timestamps >= @initial_time and timestamps <= @final_time')
    ax.plot(
        pupil_sample['timestamps'],
        pupil_sample['pupil_width']/pupil_sample['pupil_width'].max(),
        color = 'gray',
        linewidth = 1
    )
# function to plot licks
def plot_licks(ax, initial_time, final_time):
    licking_sample = dataset.licks.query('timestamps >= @initial_time and timestamps <= @final_time')
    ax.plot(
        licking_sample['timestamps'],
        np.zeros_like(licking_sample['timestamps']),
        marker = 'o',
        markersize = 3,
        color = 'black',
        linestyle = 'none'
    )
# function to plot rewards
def plot_rewards(ax, initial_time, final_time):
    rewards_sample = dataset.rewards.query('timestamps >= @initial_time and timestamps <= @final_time')
    ax.plot(
        rewards_sample['timestamps'],
        np.zeros_like(rewards_sample['timestamps']),
        marker = 'd',
        color = 'blue',
        linestyle = 'none',
        markersize = 12,
        alpha = 0.5
    )

def plot_stimuli(ax, initial_time, final_time):
    stimulus_presentations_sample = stimulus_presentations.query('end_time >= @initial_time and start_time <= @final_time')
    for idx, stimulus in stimulus_presentations_sample.iterrows():
        ax.axvspan(stimulus['start_time'], stimulus['end_time'], color=stimulus['color'], alpha=0.25)

In [ ]:
initial_time = 820 # start time in seconds
final_time = 860 # stop time in seconds
fig, ax = plt.subplots(2,1,figsize = (15,7))
plot_dff_trace(ax[0], cell_specimen_ids[3], initial_time, final_time)
plot_events_trace(ax[0], cell_specimen_ids[3], initial_time, final_time)
plot_stimuli(ax[0], initial_time, final_time)
ax[0].set_ylabel('normalized response magnitude')
ax[0].set_yticks([])
ax[0].legend(['dff trace', 'events trace'])

plot_running(ax[1], initial_time, final_time)
plot_pupil(ax[1], initial_time, final_time)
plot_licks(ax[1], initial_time, final_time)
plot_rewards(ax[1], initial_time, final_time)
plot_stimuli(ax[1], initial_time, final_time)

ax[1].set_yticks([])
ax[1].legend(['running speed', 'pupil','licks', 'rewards'])
plt.show()

### Find all ophys sessions for one animal
Using the mouse from the experiment loaded above, list every ophys session associated with it.

In [ ]:
# Get the mouse_id from the experiment we already loaded
mouse_id = dataset.metadata['mouse_id']
print('Mouse ID: {}'.format(mouse_id))

# Filter the experiment table to find all ophys sessions for this mouse
mouse_experiments = experiment_table[experiment_table.mouse_id == mouse_id]

# Get unique ophys sessions and their metadata
mouse_sessions = mouse_experiments.drop_duplicates(subset='ophys_session_id')[
    ['ophys_session_id', 'session_type', 'session_number', 'date_of_acquisition',
     'equipment_name', 'cre_line', 'targeted_structure', 'imaging_depth', 'project_code']
].sort_values(by='date_of_acquisition')

print('Number of ophys sessions for mouse {}: {}'.format(mouse_id, len(mouse_sessions)))
print('Number of ophys experiments for mouse {}: {}'.format(mouse_id, len(mouse_experiments)))
print()
mouse_sessions

### Extract aligned neural + behavioral data from one session

Load all experiments (imaging planes) for one session and merge neurons into a single N x T matrix. Within a `BehaviorOphysExperiment`, all data streams share the same hardware sync clock. We assume ophys_timestamps are identical across planes and include a check function to verify.

In [ ]:
# Pick the first ophys session for this mouse and load all its experiments (planes)
session_id = mouse_sessions.iloc[0]['ophys_session_id']
session_experiments = mouse_experiments[mouse_experiments.ophys_session_id == session_id]
print('Loading session {} ({})'.format(session_id, mouse_sessions.iloc[0]['session_type']))
print('{} imaging planes:'.format(len(session_experiments)))
print(session_experiments[['targeted_structure', 'imaging_depth']].to_string())

datasets = {}
for exp_id in session_experiments.index:
    datasets[exp_id] = bc.get_behavior_ophys_experiment(exp_id)

In [ ]:
def check_timestamp_alignment(datasets, session_experiments):
    """Check whether ophys_timestamps are identical across all planes in a session."""
    exp_ids = list(datasets.keys())
    ref_ts = datasets[exp_ids[0]].ophys_timestamps
    for eid in exp_ids[1:]:
        ts = datasets[eid].ophys_timestamps
        area = session_experiments.loc[eid, 'targeted_structure']
        depth = session_experiments.loc[eid, 'imaging_depth']
        if len(ref_ts) != len(ts):
            print('{} {}um has {} timepoints vs {} in reference'.format(
                area, depth, len(ts), len(ref_ts)))
        elif not np.array_equal(ref_ts, ts):
            max_diff = np.max(np.abs(ref_ts - ts))
            print('{} {}um timestamps differ by up to {:.6f} s'.format(
                area, depth, max_diff))
# seems unnecessary but Claude recommended this solution after reading the codebase
check_timestamp_alignment(datasets, session_experiments)

In [ ]:
# --- Common timebase ---
ref_ds = list(datasets.values())[0]
ophys_ts = ref_ds.ophys_timestamps
T = len(ophys_ts)

# --- 1) Neural data: merge dF/F across all planes into N x T ---
dff_list = []
cell_ids = []
plane_labels = []
for exp_id, ds in datasets.items():
    dff_list.append(np.vstack(ds.dff_traces.dff.values))
    cell_ids.extend(ds.dff_traces.index.tolist())
    area = session_experiments.loc[exp_id, 'targeted_structure']
    depth = session_experiments.loc[exp_id, 'imaging_depth']
    plane_labels.extend(['{}_{}um'.format(area, depth)] * len(ds.dff_traces))

neural_data = np.vstack(dff_list)

# --- 2) Running speed: interpolate from ~60 Hz to ophys timebase ---
run = ref_ds.running_speed
f_run = interp1d(run['timestamps'].values, run['speed'].values,
                 kind='linear', bounds_error=False, fill_value=np.nan)
running_speed = f_run(ophys_ts)

# --- 3) Pupil diameter: interpolate, excluding blinks ---
eye = ref_ds.eye_tracking
eye_clean = eye[~eye['likely_blink']]
f_pupil = interp1d(eye_clean['timestamps'].values, eye_clean['pupil_width'].values,
                   kind='linear', bounds_error=False, fill_value=np.nan)
pupil_diameter = f_pupil(ophys_ts)

# --- 4) Image identity at each ophys timepoint ---
stim = ref_ds.stimulus_presentations[
    ref_ds.stimulus_presentations.stimulus_block_name.str.contains('change_detection')]
stim_shown = stim[stim['image_name'] != 'omitted']

image_at_timepoint = np.full(T, '', dtype=object)
for _, row in stim_shown.iterrows():
    mask = (ophys_ts >= row['start_time']) & (ophys_ts <= row['end_time'])
    image_at_timepoint[mask] = row['image_name']

unique_images = sorted(stim_shown['image_name'].unique())
image_to_code = {name: i for i, name in enumerate(unique_images)}
image_code = np.array([image_to_code.get(img, -1) for img in image_at_timepoint])

print()
print('ophys_ts:          ({},)'.format(T))
print('neural_data:       {} neurons x {} timepoints'.format(*neural_data.shape))
print('cell_ids:          list of {}'.format(len(cell_ids)))
print('plane_labels:      list of {} — {}'.format(len(plane_labels), sorted(set(plane_labels))))
print('running_speed:     ({},)'.format(len(running_speed)))
print('pupil_diameter:    ({},)'.format(len(pupil_diameter)))
print('image_code:        ({},)  — {} unique images, -1 = blank/ISI'.format(
    len(image_code), len(unique_images)))
print('image_to_code:     {}'.format(image_to_code))

In [ ]:
### Run extraction + trial segmentation across all sessions for this mouse

def extract_session_data(bc, mouse_experiments, session_id, session_experiments):
    """Load all planes for a session, merge neural data, resample behavioral data."""
    plane_exp_ids = session_experiments.index.tolist()
    datasets = {}
    for exp_id in plane_exp_ids:
        datasets[exp_id] = bc.get_behavior_ophys_experiment(exp_id)

    ref_ds = list(datasets.values())[0]
    ophys_ts = ref_ds.ophys_timestamps
    T = len(ophys_ts)

    # Neural data: merge across planes
    dff_list = []
    cell_ids = []
    plane_labels = []
    for exp_id, ds in datasets.items():
        dff_list.append(np.vstack(ds.dff_traces.dff.values))
        cell_ids.extend(ds.dff_traces.index.tolist())
        area = session_experiments.loc[exp_id, 'targeted_structure']
        depth = session_experiments.loc[exp_id, 'imaging_depth']
        plane_labels.extend(['{}_{}um'.format(area, depth)] * len(ds.dff_traces))
    neural_data = np.vstack(dff_list)

    # Running speed
    run = ref_ds.running_speed
    f_run = interp1d(run['timestamps'].values, run['speed'].values,
                     kind='linear', bounds_error=False, fill_value=np.nan)
    running_speed = f_run(ophys_ts)

    # Pupil diameter
    eye = ref_ds.eye_tracking
    eye_clean = eye[~eye['likely_blink']]
    f_pupil = interp1d(eye_clean['timestamps'].values, eye_clean['pupil_width'].values,
                       kind='linear', bounds_error=False, fill_value=np.nan)
    pupil_diameter = f_pupil(ophys_ts)

    # Stimulus table (change detection block only)
    stim = ref_ds.stimulus_presentations[
        ref_ds.stimulus_presentations.stimulus_block_name.str.contains('change_detection')]

    return {
        'ophys_ts': ophys_ts,
        'neural_data': neural_data,
        'cell_ids': cell_ids,
        'plane_labels': plane_labels,
        'running_speed': running_speed,
        'pupil_diameter': pupil_diameter,
        'stim': stim,
    }


def segment_trials(session_data, n_frames_pre=30):
    """Segment into fixed-length trials: n_frames_pre ophys frames before each image change."""
    ophys_ts = session_data['ophys_ts']
    neural_data = session_data['neural_data']
    running_speed = session_data['running_speed']
    pupil_diameter = session_data['pupil_diameter']
    stim = session_data['stim']

    stim_shown = stim[stim['image_name'] != 'omitted'].reset_index(drop=True)
    changes = stim_shown[stim_shown['is_change'] == True]

    trials = []
    for _, change_row in changes.iterrows():
        change_time = change_row['start_time']
        frame_idx = np.searchsorted(ophys_ts, change_time) - 1
        start_idx = frame_idx - n_frames_pre + 1
        if start_idx < 0:
            continue
        idx = np.arange(start_idx, frame_idx + 1)

        pre_change_flashes = stim_shown[
            (stim_shown['end_time'] <= change_time) &
            (stim_shown['start_time'] >= ophys_ts[start_idx])]
        image_names = pre_change_flashes['image_name'].unique()
        if len(image_names) != 1:
            continue

        trials.append({
            'image_name': image_names[0],
            'neural': neural_data[:, idx],
            'running': running_speed[idx],
            'pupil': pupil_diameter[idx],
            'timestamps': ophys_ts[idx],
        })
    return trials

In [ ]:
# Loop over all ophys sessions for this mouse
unique_session_ids = mouse_sessions['ophys_session_id'].values

all_sessions = {}  # session_id -> {'session_data': ..., 'trials': ...}
for i, sid in enumerate(unique_session_ids):
    stype = mouse_sessions[mouse_sessions.ophys_session_id == sid].iloc[0]['session_type']
    sess_exps = mouse_experiments[mouse_experiments.ophys_session_id == sid]
    print('[{}/{}] Session {} ({}) — {} planes...'.format(
        i + 1, len(unique_session_ids), sid, stype, len(sess_exps)), end=' ')

    session_data = extract_session_data(bc, mouse_experiments, sid, sess_exps)
    trials = segment_trials(session_data)

    all_sessions[sid] = {
        'session_type': stype,
        'session_data': session_data,
        'trials': trials,
    }
    print('{} neurons, {} trials'.format(
        session_data['neural_data'].shape[0], len(trials)))

print('\n=== Summary ===')
for sid, s in all_sessions.items():
    n_neurons = s['session_data']['neural_data'].shape[0]
    n_trials = len(s['trials'])
    images = sorted(set(t['image_name'] for t in s['trials'])) if n_trials > 0 else []
    print('Session {} ({}): {} neurons, {} trials, images: {}'.format(
        sid, s['session_type'], n_neurons, n_trials, images))

### Check neurons that are present across all recording sessions

In [ ]:
# Check neuron identity matching across sessions using cell_specimen_id
# cell_ids are already cell_specimen_ids (the cross-session matched ID)
session_cell_sets = {}
for sid, s in all_sessions.items():
    session_cell_sets[sid] = set(s['session_data']['cell_ids'])

# Find neurons present in ALL sessions
all_sids = list(session_cell_sets.keys())
shared_all = session_cell_sets[all_sids[0]]
for sid in all_sids[1:]:
    shared_all = shared_all & session_cell_sets[sid]

print('Neurons per session:')
for sid, s in all_sessions.items():
    print('  {} ({}): {} neurons'.format(sid, s['session_type'],
        len(session_cell_sets[sid])))

print('Neurons present in ALL {} sessions: {}'.format(
    len(all_sids), len(shared_all)))